# HAI Causal Graph Builder

Builds the 3-level causal graph used by `causal_loss.py`.

| Level | Meaning |
|---|---|
| L0 | sensor → actuator (DCS control loops) |
| L1 | actuator → sensor (1-hop physical) |
| L2 | actuator → element → sensor (2-hop) |

## 1 — Setup: paths, imports, sensor map

In [ ]:
from __future__ import annotations
import ast, csv, json
from collections import defaultdict
from datetime import datetime
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

BOILER_DIR  = Path("C:/Users/ahmma/Desktop/farah/boiler")
# Resolve repo root regardless of CWD (notebook may run from 01_causal_graph/ or repo root)
_cwd = Path().resolve()
ROOT = _cwd if (_cwd / "data/processed").exists() else _cwd.parent
DATA_DIR    = ROOT / "data/processed"
OUT_DIR     = ROOT / "outputs/causal_graph"
TRAIN_CSVS  = [DATA_DIR / f'train{i}.csv' for i in range(1, 5)]
TDMI_SAMPLE = 200_000
LAG_MAX     = 30
N_BINS      = 10
MIN_STD     = 1e-6
DYNAMICS_LAG_RANGE = {1: (1, 5), 2: (5, 15), 3: (15, 30)}

boiler_files = sorted(BOILER_DIR.glob('dcs_*.json'))

# ── Step 1a: Build combined graph (DCS + physical) using raw short names ──
G_combined = nx.DiGraph()

for json_path in boiler_files:
    d = ast.literal_eval(json_path.read_text(encoding='utf-8'))
    for n in d['nodes']:
        G_combined.add_node(n['id'], fillcolor=n.get('fillcolor', 'white'), source='dcs')
    for link in d['links']:
        G_combined.add_edge(link['source'], link['target'], source='dcs')

phy_data_raw = json.loads((BOILER_DIR / 'phy_boiler.json').read_text(encoding='utf-8'))
for n in phy_data_raw['nodes']:
    if n['id'] not in G_combined:
        G_combined.add_node(n['id'], source='physical')
for link in phy_data_raw['links']:
    G_combined.add_edge(link['source'], link['target'], source='physical')

print(f"Combined graph: {G_combined.number_of_nodes()} nodes, {G_combined.number_of_edges()} edges")

# ── Step 1b: Auto-generate SENSOR_MAP by matching short names to CSV columns ─
# Short names = yellow DCS nodes + phy actuator/sensor nodes
yellow_nodes = {n for n, d in G_combined.nodes(data=True) if d.get('fillcolor') == 'yellow'}
phy_io_nodes = set()
for n in phy_data_raw['nodes']:
    in_t  = str(n.get('in_tags',  '') or '').strip()
    out_t = str(n.get('out_tags', '') or '').strip()
    if in_t or out_t:
        phy_io_nodes.add(n['id'])
short_names = yellow_nodes | phy_io_nodes
print(f"Short names to map: {len(short_names)}")

# Read CSV column headers from train1.csv
train1 = DATA_DIR / 'train1.csv'
with open(train1, newline='', encoding='utf-8') as f:
    csv_cols = next(csv.reader(f))
print(f"CSV columns: {len(csv_cols)}")

# Manual overrides for names that differ between JSON and CSV
OVERRIDES = {'TWIT03': 'P1_TIT03', 'LSH01': 'P1_PIT01_HH', 'LSL01': 'P1_SOL01D'}

# Match: short_name → csv_col if short_name appears inside csv_col (case-insensitive)
def _best_col(short, csv_cols):
    matches = [col for col in csv_cols if short.upper() in col.upper()]
    if not matches: return None
    if len(matches) == 1: return matches[0]
    # Prefer P1_ prefix over DM- and other prefixes
    p1 = [col for col in matches if col.startswith('P1_')]
    pool = p1 if p1 else matches
    # Among candidates, prefer exact suffix then exact suffix+D then shortest
    su = short.upper()
    exact_d = [col for col in pool if col.upper().endswith(su + 'D')]
    if exact_d: return exact_d[0]
    exact   = [col for col in pool if col.upper().endswith(su)]
    if exact: return exact[0]
    return min(pool, key=len)

SENSOR_MAP = dict(OVERRIDES)
for short in sorted(short_names):
    if short in SENSOR_MAP: continue
    col = _best_col(short, csv_cols)
    if col: SENSOR_MAP[short] = col

HAI_TO_SHORT = {v: k for k, v in SENSOR_MAP.items()}

print(f"Auto-mapped {len(SENSOR_MAP)} short names to HAI columns (+ {len(OVERRIDES)} manual overrides)")
print(f"Sample: {list(SENSOR_MAP.items())[:8]}")
print(f"Unmapped short names: {sorted(short_names - set(SENSOR_MAP))}")
print(f"Boiler JSON files ({len(boiler_files)}): {[f.name for f in boiler_files]}")
print(f"Train CSVs: {[str(p) for p in TRAIN_CSVS]}")

## 2 — L0: Extract DCS control relationships

BFS through PLC blocks to find sensor→actuator relationships from `dcs_*.json`.

In [ ]:
def _load_dcs(path): return ast.literal_eval(path.read_text(encoding='utf-8'))
def _yellow(d): return {n['id'] for n in d['nodes'] if n.get('fillcolor')=='yellow'}
def _grey(d):   return {n['id'] for n in d['nodes'] if n.get('fillcolor')=='lightgrey'}
def _adj(d):
    a = defaultdict(list)
    for e in d['links']: a[e['source']].append(e['target'])
    return a

def _bfs(src, adj, yellow, grey):
    visited, queue, reached = {src}, [src], set()
    while queue:
        node = queue.pop(0)
        for nb in adj.get(node, []):
            if nb in visited: continue
            visited.add(nb)
            if nb in yellow: reached.add(nb)
            elif nb not in grey: queue.append(nb)
    return reached

raw_edges, all_yellow = [], set()
for path in boiler_files:
    d = _load_dcs(path)
    yellow, grey, adj = _yellow(d), _grey(d), _adj(d)
    all_yellow.update(yellow)
    count = 0
    for src in yellow:
        for tgt in _bfs(src, adj, yellow, grey):
            if src != tgt: raw_edges.append((src, tgt, path.stem)); count += 1
    print(f"  {path.stem:12s}: {len(yellow):3d} yellow | {count:3d} raw edges")

seen, l0_edges = set(), []
for src, tgt, plc in raw_edges:
    sh, th = SENSOR_MAP.get(src), SENSOR_MAP.get(tgt)
    if sh and th and sh != th and (sh, th) not in seen:
        seen.add((sh, th))
        l0_edges.append({'source_hai': sh, 'target_hai': th, 'level': 0, 'dynamics': 1, 'via': plc})

print(f"Total yellow: {len(all_yellow)}  |  Raw edges: {len(raw_edges)}  |  L0 mapped: {len(l0_edges)}")
print("L0 edges (first 5):")
for e in l0_edges[:5]:
    print(f"  {e['source_hai']} -> {e['target_hai']} (via {e['via']})")

## 3 — Load physical graph (`phy_boiler.json`)

Classifies nodes as actuator / sensor / element and visualises a sample subgraph.

In [ ]:
phy_data = json.loads((BOILER_DIR / "phy_boiler.json").read_text(encoding="utf-8"))

node_role = {}
for n in phy_data['nodes']:
    nid  = n['id']
    in_t = str(n.get('in_tags',  '') or '').strip()
    out_t= str(n.get('out_tags', '') or '').strip()
    node_role[nid] = 'actuator' if in_t else ('sensor' if out_t else 'element')

phy_adj = defaultdict(list)
for e in phy_data['links']:
    dyn = int(e['dynamics']) if str(e.get('dynamics', '')).isdigit() else 1
    phy_adj[e['source']].append((e['target'], dyn))
phy_adj = dict(phy_adj)

n_act = sum(1 for r in node_role.values() if r == 'actuator')
n_sen = sum(1 for r in node_role.values() if r == 'sensor')
n_ele = sum(1 for r in node_role.values() if r == 'element')
print(f"PHY nodes: {len(node_role)}  (actuators={n_act}, sensors={n_sen}, elements={n_ele})")
print(f"PHY edges: {sum(len(v) for v in phy_adj.values())}")
print("Sample:")
for src, targets in list(phy_adj.items())[:3]:
    for tgt, dyn in targets[:2]:
        print(f"  {src} -> {tgt} (dyn={dyn})")

# Visualise sample subgraph
G_phy = nx.DiGraph()
snodes = list(node_role.keys())[:12]
for src in snodes:
    for tgt, _ in phy_adj.get(src, []):
        if tgt in snodes: G_phy.add_edge(src, tgt)

cmap = {'actuator': '#ff7f0e', 'sensor': '#2ca02c', 'element': '#aec7e8'}
nc   = [cmap.get(node_role.get(n, 'element'), 'gray') for n in G_phy.nodes()]
plt.figure(figsize=(10, 6))
pos = nx.spring_layout(G_phy, seed=42)
nx.draw(G_phy, pos, with_labels=True, node_color=nc, node_size=1200, font_size=7, arrows=True, arrowsize=15)
from matplotlib.patches import Patch
plt.legend(handles=[Patch(color=c, label=l) for l, c in cmap.items()], loc='upper left')
plt.title('Physical graph (sample — first 12 nodes)')
plt.tight_layout(); plt.show()

## 4 — Build L1 and L2 physical chains

**L1**: actuator directly affects sensor (1 hop)

**L2**: actuator → element → sensor (2 hops)

In [ ]:
l12_edges = []
seen_l12  = {(e['source_hai'], e['target_hai']) for e in l0_edges}

for act_hai in set(SENSOR_MAP.values()):
    act_short = HAI_TO_SHORT.get(act_hai)
    if not act_short or act_short not in phy_adj: continue
    for nb1_short, dyn1 in phy_adj[act_short]:
        nb1_hai = SENSOR_MAP.get(nb1_short)
        if nb1_hai and nb1_hai != act_hai:
            key = (act_hai, nb1_hai)
            if key not in seen_l12:
                seen_l12.add(key)
                l12_edges.append({'source_hai': act_hai, 'target_hai': nb1_hai,
                                   'level': 1, 'dynamics': dyn1, 'via': 'phy_boiler'})
        elif nb1_hai is None:
            for nb2_short, dyn2 in phy_adj.get(nb1_short, []):
                nb2_hai = SENSOR_MAP.get(nb2_short)
                if nb2_hai and nb2_hai != act_hai:
                    key = (act_hai, nb2_hai)
                    if key not in seen_l12:
                        seen_l12.add(key)
                        l12_edges.append({'source_hai': act_hai, 'target_hai': nb2_hai,
                                          'level': 2, 'dynamics': max(dyn1, dyn2),
                                          'via': f'phy_boiler:{nb1_short}'})

l1 = [e for e in l12_edges if e['level'] == 1]
l2 = [e for e in l12_edges if e['level'] == 2]
print(f"L1 edges: {len(l1)}  |  L2 edges: {len(l2)}")
print("L1 sample (actuator -> sensor, 1-hop):")
for e in l1[:3]:
    print(f"  {e['source_hai']} -> {e['target_hai']}")
print("L2 sample (actuator -> element -> sensor, 2-hop):")
for e in l2[:3]:
    elem = e['via'].split(':')[-1]
    print(f"  {e['source_hai']} -> [{elem}] -> {e['target_hai']}")

# Visualise one L2 example
if l2:
    ex   = l2[0]
    elem = ex['via'].split(':')[-1]
    G_ex = nx.DiGraph()
    G_ex.add_edges_from([(ex['source_hai'], elem), (elem, ex['target_hai'])])
    plt.figure(figsize=(6, 3))
    nx.draw(G_ex, nx.planar_layout(G_ex), with_labels=True,
            node_color=['#ff7f0e', '#aec7e8', '#2ca02c'],
            node_size=2000, font_size=8, arrows=True, arrowsize=20)
    plt.title(f"L2 example: {ex['source_hai']} -> {elem} -> {ex['target_hai']}")
    plt.tight_layout(); plt.show()

## 5 — Estimate lags via TDMI / cross-correlation

For each (parent, child) pair, finds the time lag that maximises mutual information on differenced signals.

In [ ]:
def _load_cols(csv_paths, cols, max_rows):
    per_file = max(1, max_rows // len(csv_paths))
    data = {c: [] for c in cols}
    for path in csv_paths:
        if not path.exists(): continue
        rows = 0
        with open(path, newline='', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                if rows >= per_file: break
                for c in cols:
                    v = row.get(c)
                    if v is None: continue
                    try: data[c].append(float(v))
                    except ValueError: data[c].append(float('nan'))
                rows += 1
        print(f"    {path.name}: {rows:,} rows")
    return {c: np.array(v, dtype=np.float64) for c, v in data.items() if v}

def _tdmi(x, y, lag, nb=10):
    if lag <= 0 or lag >= len(x): return 0.0
    xp, yp = x[:-lag], y[lag:]
    mask = ~(np.isnan(xp) | np.isnan(yp))
    xv, yv = xp[mask], yp[mask]
    if len(xv) < 2*nb or xv.max() == xv.min() or yv.max() == yv.min(): return 0.0
    xe = np.linspace(xv.min(), xv.max()+1e-10, nb+1)
    ye = np.linspace(yv.min(), yv.max()+1e-10, nb+1)
    xb = np.clip(np.digitize(xv, xe)-1, 0, nb-1)
    yb = np.clip(np.digitize(yv, ye)-1, 0, nb-1)
    j  = np.zeros((nb, nb))
    np.add.at(j, (xb, yb), 1); j /= j.sum()
    px, py = j.sum(axis=1, keepdims=True), j.sum(axis=0, keepdims=True)
    pos = j > 0
    return max(float(np.sum(j[pos] * np.log(j[pos] / (px*py)[pos]))), 0.0)

def _xclag(x, y, lo, hi):
    dx, dy = np.diff(x.astype(float)), np.diff(y.astype(float))
    mask = ~(np.isnan(dx) | np.isnan(dy)); dx, dy = dx[mask], dy[mask]
    if len(dx) < hi + 10: return (lo+hi)//2
    bl, bc = lo, -np.inf
    for lag in range(lo, hi+1):
        if lag >= len(dx): break
        c = float(np.corrcoef(dx[:-lag], dy[lag:])[0,1]) if lag > 0 else float(np.corrcoef(dx, dy)[0,1])
        if abs(c) > abs(bc): bc, bl = c, lag
    return bl

def _best_lag(x, y, dyn):
    lo, hi = DYNAMICS_LAG_RANGE.get(dyn, (1, LAG_MAX))
    mid = (lo+hi)//2
    if np.nanstd(x) < MIN_STD or np.nanstd(y) < MIN_STD: return mid, 'constant_fallback'
    dx, dy = np.diff(x), np.diff(y)
    lags   = list(range(lo, hi+1))
    mi     = [_tdmi(dx, dy, lag, N_BINS) for lag in lags]
    if max(mi) >= 1e-5: return lags[int(np.argmax(mi))], 'tdmi'
    return _xclag(x, y, lo, hi), 'xcorr_fallback'

all_edges = l0_edges + l12_edges
needed    = set()
for e in all_edges: needed.update([e['source_hai'], e['target_hai']])
print(f"Loading data for {len(needed)} columns...")
col_data = _load_cols(TRAIN_CSVS, needed, TDMI_SAMPLE)
print(f"Columns loaded: {len(col_data)}")

n_tdmi = n_const = n_xcorr = 0
for e in all_edges:
    x, y = col_data.get(e['source_hai']), col_data.get(e['target_hai'])
    if x is None or y is None or len(x) == 0 or len(y) == 0:
        lo, hi = DYNAMICS_LAG_RANGE.get(e['dynamics'], (1, LAG_MAX))
        e['lag'], e['lag_method'] = (lo+hi)//2, 'missing_column'; n_const += 1; continue
    lag, method = _best_lag(x, y, e['dynamics'])
    e['lag'], e['lag_method'] = lag, method
    if method == 'tdmi': n_tdmi += 1
    elif method == 'constant_fallback': n_const += 1
    else: n_xcorr += 1

print(f"Lag methods: tdmi={n_tdmi}, constant_fallback={n_const}, xcorr_fallback={n_xcorr}")
lag_vals = [e['lag'] for e in all_edges]
plt.figure(figsize=(8, 4))
plt.hist(lag_vals, bins=range(0, max(lag_vals)+3, 1), color='steelblue', edgecolor='white')
plt.xlabel('Lag (seconds)'); plt.ylabel('Number of edges')
plt.title('Distribution of estimated lags across all causal edges')
plt.tight_layout(); plt.show()

## 6 — Save outputs

Writes `parents_full.json`, `edges_full.csv`, `summary_full.txt` to `outputs/causal_graph/`.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Filter: keep only edges whose HAI columns are real sensor/actuator cols (P1_ / P4_)
def _is_hai(col): return col.startswith('P1_') or col.startswith('P4_')
all_edges = [e for e in all_edges if _is_hai(e['source_hai']) and _is_hai(e['target_hai'])]
print(f"After filtering DM-/non-HAI columns: {len(all_edges)} edges")

# parents_full.json
parents = {}
for e in all_edges:
    parents.setdefault(e['target_hai'], []).append({
        'parent': e['source_hai'], 'lag': e['lag'],
        'level': e['level'], 'dynamics': e['dynamics'],
        'via': e['via'], 'lag_method': e['lag_method'],
    })
with open(OUT_DIR / 'parents_full.json', 'w') as f:
    json.dump(parents, f, indent=2)

# edges_full.csv
fieldnames = ['source_hai', 'target_hai', 'level', 'dynamics', 'lag', 'lag_method', 'via']
with open(OUT_DIR / 'edges_full.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
    w.writeheader()
    w.writerows(sorted(all_edges, key=lambda e: (e['level'], e['source_hai'])))

# summary_full.txt
l0 = [e for e in all_edges if e['level'] == 0]
l1 = [e for e in all_edges if e['level'] == 1]
l2 = [e for e in all_edges if e['level'] == 2]
methods = defaultdict(int)
for e in all_edges: methods[e.get('lag_method', '?')] += 1
lines = [
    '=== HAI Digital Twin - Full Causal Graph ===',
    f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}',
    '', 'Edge counts:',
    f'  L0 : {len(l0)}', f'  L1 : {len(l1)}', f'  L2 : {len(l2)}',
    f'  Total : {len(all_edges)}',
    '', 'Lag breakdown:',
    *[f'  {m:25s}: {c}' for m, c in sorted(methods.items())],
]
(OUT_DIR / 'summary_full.txt').write_text('\n'.join(lines), encoding='utf-8')

print(f"Outputs saved to {OUT_DIR}/")
print(f"  parents_full.json — {len(parents)} target sensors, {len(all_edges)} total edges")
print(f"  edges_full.csv")
print(f"  summary_full.txt")
print(f"Edge breakdown:  L0={len(l0)}  L1={len(l1)}  L2={len(l2)}")

---
## Case Study I — Attack Propagation Chain (APC)

Forward analysis from an attack entry point.
Red = entry, orange = affected sensors/actuators, blue = PLC blocks.

In [ ]:
# ── Build full DCS graph (all dcs_*.json merged) ─────────────────────────
import ast
G_dcs = nx.DiGraph()
for path in boiler_files:
    d = ast.literal_eval(path.read_text(encoding='utf-8'))
    for n in d['nodes']:
        G_dcs.add_node(n['id'], fillcolor=n.get('fillcolor', 'white'))
    for e in d['links']:
        G_dcs.add_edge(e['source'], e['target'], label=e.get('label', ''))

print(f"DCS graph: {G_dcs.number_of_nodes()} nodes, {G_dcs.number_of_edges()} edges")

# ── Case I: Attack Propagation Chain (forward analysis) ───────────────────
# Entry point: 1001.7 (HMI keyboard manipulation, as described in PDF Fig.16)
ATTACK_ENTRY = '1001.7'
if ATTACK_ENTRY in G_dcs:
    reachable = nx.descendants(G_dcs, ATTACK_ENTRY)
    apc_nodes = {ATTACK_ENTRY} | reachable
    apc_subgraph = G_dcs.subgraph(apc_nodes)
    print(f"Case I — APC from {ATTACK_ENTRY}:")
    print(f"  Reachable nodes ({len(reachable)}): {sorted(reachable)}")
    # Show all simple paths to yellow (sensor/actuator) nodes
    yellow_nodes = [n for n, d in G_dcs.nodes(data=True) if d.get('fillcolor') == 'yellow' and n in reachable]
    print(f"  Affected sensor/actuator nodes: {yellow_nodes}")
    # Plot the APC subgraph
    plt.figure(figsize=(14, 8))
    pos = nx.spring_layout(apc_subgraph, seed=42, k=2)
    node_colors = ['#d62728' if n == ATTACK_ENTRY else
                   '#ff7f0e' if G_dcs.nodes[n].get('fillcolor') == 'yellow' else
                   '#aec7e8' for n in apc_subgraph.nodes()]
    nx.draw(apc_subgraph, pos, with_labels=True, node_color=node_colors,
            node_size=800, font_size=7, arrows=True, arrowsize=12)
    from matplotlib.patches import Patch
    plt.legend(handles=[
        Patch(color='#d62728', label='Attack entry'),
        Patch(color='#ff7f0e', label='Sensor/Actuator (affected)'),
        Patch(color='#aec7e8', label='PLC block'),
    ])
    plt.title(f'Case I: Attack Propagation Chain from {ATTACK_ENTRY} (HMI keyboard)')
    plt.tight_layout(); plt.show()
else:
    print(f"Node {ATTACK_ENTRY} not found in DCS graph — check boiler_files")

## Case Study II — Root-cause Analysis

Backward analysis from observed anomaly (PIT01 pressure sensor).
Green = observed anomaly, red = root-cause candidates.

In [ ]:
# ── Case II: Root-cause analysis (backward analysis) ─────────────────────
# Anomaly observed at PIT01 (pressure sensor) — trace back to find root cause
# Find DCS node whose edge label contains 'PIT01'
pit01_nodes = [n for n in G_dcs.nodes() if 'PIT01' in n]
# Also find via edge labels
for u, v, data in G_dcs.edges(data=True):
    if 'PIT01' in data.get('label', ''):
        pit01_nodes.append(v)
pit01_nodes = list(set(pit01_nodes))
print(f"Nodes associated with PIT01: {pit01_nodes}")

# Use first match as anomaly observation point
if pit01_nodes:
    OBS_NODE = pit01_nodes[0]
    G_dcs_rev = G_dcs.reverse()
    ancestors = nx.descendants(G_dcs_rev, OBS_NODE)
    root_candidates = [n for n in ancestors if G_dcs_rev.in_degree(n) == 0 or G_dcs.in_degree(n) == 0]
    print(f"Case II — Root-cause analysis for anomaly at {OBS_NODE}:")
    print(f"  Ancestors ({len(ancestors)}): {sorted(ancestors)}")
    print(f"  Root-cause candidates (source nodes): {root_candidates}")
    # Plot backward reachability
    back_nodes = {OBS_NODE} | ancestors
    back_sg = G_dcs.subgraph(back_nodes)
    plt.figure(figsize=(14, 8))
    pos = nx.spring_layout(back_sg, seed=7, k=2)
    node_colors = ['#2ca02c' if n == OBS_NODE else
                   '#d62728' if n in root_candidates else
                   '#aec7e8' for n in back_sg.nodes()]
    nx.draw(back_sg, pos, with_labels=True, node_color=node_colors,
            node_size=800, font_size=7, arrows=True, arrowsize=12)
    from matplotlib.patches import Patch
    plt.legend(handles=[
        Patch(color='#2ca02c', label='Observed anomaly (PIT01)'),
        Patch(color='#d62728', label='Root-cause candidates'),
        Patch(color='#aec7e8', label='Intermediate nodes'),
    ])
    plt.title(f'Case II: Root-cause analysis — backward from {OBS_NODE}')
    plt.tight_layout(); plt.show()

## Case Study III — Physical Subsystem Graph

Full `phy_boiler.json` visualised with hydrodynamic vs thermodynamic edge classification.

In [ ]:
# ── Physical subsystem graph (phy_boiler.json) ───────────────────────────
import json
phy_raw = json.loads((BOILER_DIR / 'phy_boiler.json').read_text(encoding='utf-8'))

G_phy_full = nx.DiGraph()
for n in phy_raw['nodes']:
    # Parse out_tags to find HAI sensor column name
    out_t = str(n.get('out_tags', '') or '').strip()
    in_t  = str(n.get('in_tags',  '') or '').strip()
    role  = 'actuator' if in_t else ('sensor' if out_t else 'element')
    G_phy_full.add_node(n['id'], role=role, desc=n.get('desc',''), out_tags=out_t, in_tags=in_t)

for e in phy_raw['links']:
    dyn = e.get('dynamics', 1)
    G_phy_full.add_edge(e['source'], e['target'], dynamics=dyn)

print(f"Physical graph: {G_phy_full.number_of_nodes()} nodes, {G_phy_full.number_of_edges()} edges")
roles = {r: sum(1 for _,d in G_phy_full.nodes(data=True) if d.get('role')==r) for r in ['actuator','sensor','element']}
print(f"  Roles: {roles}")

# Node labels: id + desc
labels = {n: f"{n}\n{d.get('desc','')[:12]}" for n,d in G_phy_full.nodes(data=True)}
cmap_phy = {'actuator': '#ff7f0e', 'sensor': '#2ca02c', 'element': '#aec7e8'}
nc_phy   = [cmap_phy.get(G_phy_full.nodes[n].get('role','element'), 'gray') for n in G_phy_full.nodes()]

plt.figure(figsize=(16, 10))
pos = nx.spring_layout(G_phy_full, seed=0, k=3)
nx.draw(G_phy_full, pos, labels=labels, node_color=nc_phy,
        node_size=1400, font_size=6, arrows=True, arrowsize=12)
from matplotlib.patches import Patch
plt.legend(handles=[Patch(color=c, label=l) for l, c in cmap_phy.items()], loc='upper left')
plt.title('Physical subsystem graph (full phy_boiler.json)')
plt.tight_layout(); plt.show()

# Hydrodynamic vs thermodynamic edges
hydro_edges = [(u,v) for u,v,d in G_phy_full.edges(data=True) if d.get('dynamics',1) <= 2]
thermo_edges= [(u,v) for u,v,d in G_phy_full.edges(data=True) if d.get('dynamics',1) >= 3]
print(f"  Hydrodynamic edges (dyn<=2): {len(hydro_edges)}")
print(f"  Thermodynamic edges (dyn>=3): {len(thermo_edges)}")